# 💰 Monthly Spending Data Analysis (2020–2025)
### A comprehensive personal finance analysis using Python
---

## 1. Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='muted')

# ── Load Dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('monthly_spending_dataset_2020_2025.csv', parse_dates=['Month'])
df.rename(columns=lambda c: c.strip(), inplace=True)

print('Dataset loaded successfully!')
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 2. Dataset Overview & Quality Check

In [ ]:
print('='*60)
print('DATASET INFO')
print('='*60)
df.info()

print('\n' + '='*60)
print('MISSING VALUES')
print('='*60)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found ✅')

print('\n' + '='*60)
print('DUPLICATE ROWS')
print('='*60)
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}' if dupes > 0 else 'No duplicate rows ✅')

print('\n' + '='*60)
print('DATA TYPES')
print('='*60)
print(df.dtypes)

print('\n' + '='*60)
print('NEGATIVE / ZERO VALUES CHECK')
print('='*60)
numeric_cols = df.select_dtypes(include='number').columns
neg_check = (df[numeric_cols] < 0).sum()
print(neg_check[neg_check > 0] if neg_check.any() else 'No negative values ✅')

In [ ]:
print('DESCRIPTIVE STATISTICS')
df.describe().round(2)

## 3. Feature Engineering

In [ ]:
# ── Derived Time Columns ──────────────────────────────────────────────────────
df['Year']       = df['Month'].dt.year
df['MonthNum']   = df['Month'].dt.month
df['MonthName']  = df['Month'].dt.strftime('%b')
df['YearMonth']  = df['Month'].dt.to_period('M')

# ── Category Columns (spending categories) ────────────────────────────────────
CATEGORY_COLS = [
    'Groceries (₹)', 'Rent (₹)', 'Transportation (₹)', 'Gym (₹)',
    'Utilities (₹)', 'Healthcare (₹)', 'Investments (₹)', 'Savings (₹)',
    'EMI/Loans (₹)', 'Dining & Entertainment (₹)', 'Shopping & Wants (₹)'
]

SHORT_NAMES = [
    'Groceries', 'Rent', 'Transportation', 'Gym',
    'Utilities', 'Healthcare', 'Investments', 'Savings',
    'EMI/Loans', 'Dining & Ent.', 'Shopping'
]

# Rename for display convenience
rename_map = dict(zip(CATEGORY_COLS, SHORT_NAMES))
df_clean = df.rename(columns=rename_map)
df_clean.rename(columns={
    'Total Expenditure (₹)': 'Total_Expenditure',
    'Income (₹)': 'Income'
}, inplace=True)

CAT_COLS = SHORT_NAMES.copy()

# ── Step 3 Task: Calculate Total Spending = sum of all categories ─────────────
# (Dataset already has Total Expenditure; we validate by recalculating)
df_clean['Calculated_Total'] = df_clean[CAT_COLS].sum(axis=1)
df_clean['Total_Check_OK']   = np.isclose(df_clean['Calculated_Total'], df_clean['Total_Expenditure'])

print('Total Spending Validation (Calculated vs Provided):')
print(df_clean[['Month','Calculated_Total','Total_Expenditure','Total_Check_OK']].head(10))
print(f"\nAll rows match: {df_clean['Total_Check_OK'].all()}")

# ── Savings Rate & Discretionary Spending ─────────────────────────────────────
df_clean['Savings_Rate']       = (df_clean['Savings'] / df_clean['Income'] * 100).round(2)
df_clean['Discretionary']      = df_clean['Dining & Ent.'] + df_clean['Shopping']
df_clean['Essential']          = (
    df_clean['Groceries'] + df_clean['Rent'] + df_clean['Transportation'] +
    df_clean['Utilities'] + df_clean['Healthcare'] + df_clean['EMI/Loans']
)
df_clean['Investment_Savings'] = df_clean['Investments'] + df_clean['Savings']

print('\nFeature engineering complete ✅')
df_clean[['Month','Year','Savings_Rate','Discretionary','Essential','Investment_Savings']].head()

## 4. Category-wise Analysis

In [ ]:
# ── Total spending per category ────────────────────────────────────────────────
cat_totals   = df_clean[CAT_COLS].sum().sort_values(ascending=False)
cat_means    = df_clean[CAT_COLS].mean().sort_values(ascending=False)
cat_pct      = (cat_totals / cat_totals.sum() * 100).round(2)

cat_summary = pd.DataFrame({
    'Total (₹)':   cat_totals,
    'Monthly Avg (₹)': cat_means.round(2),
    'Share (%)':   cat_pct
})

print('='*60)
print('CATEGORY SUMMARY (All-Time)')
print('='*60)
print(cat_summary.to_string())

In [ ]:
# ── Highest & Lowest spending categories ──────────────────────────────────────
print('HIGHEST spending category:', cat_totals.idxmax(), f'— ₹{cat_totals.max():,.0f}')
print('LOWEST  spending category:', cat_totals.idxmin(), f'— ₹{cat_totals.min():,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
ax = axes[0]
bars = ax.barh(cat_totals.index, cat_totals.values, color=sns.color_palette('muted', len(cat_totals)))
ax.set_xlabel('Total Spending (₹)')
ax.set_title('Total Spending by Category (2020–2025)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.1f}L'))
for bar in bars:
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
            f'₹{bar.get_width()/1000:.0f}K', va='center', fontsize=9)

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    cat_totals.values, labels=cat_totals.index, autopct='%1.1f%%',
    startangle=140, pctdistance=0.82,
    colors=sns.color_palette('muted', len(cat_totals))
)
for t in autotexts:
    t.set_fontsize(8)
ax2.set_title('Spending Distribution by Category', fontweight='bold')

plt.tight_layout()
plt.savefig('chart_category_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Monthly Total & Average Spending

In [ ]:
# ── Monthly totals ────────────────────────────────────────────────────────────
monthly = df_clean[['Month','Year','Total_Expenditure','Income','Savings_Rate']].copy()
monthly['Surplus'] = monthly['Income'] - monthly['Total_Expenditure']

print('Monthly Total & Average Spending')
print(f"  Overall Monthly Average: ₹{monthly['Total_Expenditure'].mean():,.2f}")
print(f"  Overall Monthly Max:     ₹{monthly['Total_Expenditure'].max():,.2f}  ({monthly.loc[monthly['Total_Expenditure'].idxmax(),'Month'].strftime('%b %Y')})")
print(f"  Overall Monthly Min:     ₹{monthly['Total_Expenditure'].min():,.2f}  ({monthly.loc[monthly['Total_Expenditure'].idxmin(),'Month'].strftime('%b %Y')})")

# Yearly summary
yearly_summary = df_clean.groupby('Year').agg(
    Total_Expenditure=('Total_Expenditure','sum'),
    Avg_Monthly_Expenditure=('Total_Expenditure','mean'),
    Total_Income=('Income','sum'),
    Avg_Savings_Rate=('Savings_Rate','mean')
).round(2)
print('\nYearly Summary:')
print(yearly_summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Monthly total expenditure line
ax1 = axes[0]
ax1.plot(monthly['Month'], monthly['Total_Expenditure'], marker='o', linewidth=2,
         color='steelblue', markersize=4, label='Total Expenditure')
ax1.plot(monthly['Month'], monthly['Income'], linestyle='--', linewidth=1.5,
         color='green', label='Income')
ax1.fill_between(monthly['Month'], monthly['Total_Expenditure'], monthly['Income'],
                 alpha=0.12, color='green', label='Surplus')
ax1.set_title('Monthly Total Expenditure vs Income (2020–2025)', fontweight='bold')
ax1.set_ylabel('Amount (₹)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax1.legend()

# Savings rate
ax2 = axes[1]
ax2.bar(monthly['Month'], monthly['Savings_Rate'], color='teal', alpha=0.7, width=20)
ax2.axhline(monthly['Savings_Rate'].mean(), color='red', linestyle='--', linewidth=1.5,
            label=f"Avg: {monthly['Savings_Rate'].mean():.1f}%")
ax2.set_title('Monthly Savings Rate (%)', fontweight='bold')
ax2.set_ylabel('Savings Rate (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('chart_monthly_expenditure.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Spending Analysis by Payment Method & Spending Type

In [ ]:
# ── Spending Type Classification ──────────────────────────────────────────────
# Essential vs Discretionary vs Investment/Savings
spending_type_totals = pd.Series({
    'Essential\n(Groceries+Rent+Transport\n+Utilities+Healthcare+EMI)': df_clean['Essential'].sum(),
    'Investment\n& Savings':  df_clean['Investment_Savings'].sum(),
    'Discretionary\n(Dining+Shopping)': df_clean['Discretionary'].sum(),
    'Gym': df_clean['Gym'].sum()
})

print('Spending Type Breakdown (All-Time Totals):')
for k, v in spending_type_totals.items():
    pct = v / spending_type_totals.sum() * 100
    print(f'  {k.replace(chr(10)," "):<55}  ₹{v:>12,.0f}   ({pct:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#4c72b0', '#55a868', '#c44e52', '#dd8452']
bars = ax.bar(
    ['Essential', 'Investment & Savings', 'Discretionary', 'Gym'],
    spending_type_totals.values,
    color=colors, edgecolor='white', linewidth=1.2
)
ax.set_title('Total Spending by Type (2020–2025)', fontweight='bold')
ax.set_ylabel('Total Spending (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e6:.1f}M'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30000,
            f'₹{bar.get_height()/1e5:.1f}L', ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('chart_spending_type.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Group & Summarize – Year/Month Aggregations

In [ ]:
# ── Yearly totals per category ────────────────────────────────────────────────
yearly_cat = df_clean.groupby('Year')[CAT_COLS].sum()
print('Yearly Total per Category (₹):')
print(yearly_cat.to_string())

# Plot
yearly_cat.plot(kind='bar', figsize=(16, 7), colormap='tab20', edgecolor='white')
plt.title('Yearly Spending per Category', fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Total Spending (₹)')
plt.legend(loc='upper left', fontsize=9, ncol=2)
plt.xticks(rotation=0)
plt.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
plt.tight_layout()
plt.savefig('chart_yearly_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Month-of-year average (seasonal patterns) ─────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
seasonal = df_clean.groupby('MonthName')['Total_Expenditure'].mean().reindex(month_order)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(seasonal.index, seasonal.values, color=sns.color_palette('coolwarm', 12), edgecolor='white')
ax.axhline(seasonal.mean(), color='red', linestyle='--', label=f'Mean: ₹{seasonal.mean():,.0f}')
ax.set_title('Average Monthly Expenditure by Month-of-Year (Seasonal Pattern)', fontweight='bold')
ax.set_ylabel('Average Expenditure (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.savefig('chart_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()

print('Average spending by month of year:')
print(seasonal.round(2).to_string())

## 8. Heatmap – Category Spending Across Years

In [ ]:
heatmap_data = df_clean.groupby('Year')[CAT_COLS].mean().T

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Monthly (₹)'}
)
ax.set_title('Heatmap: Average Monthly Category Spending per Year', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig('chart_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Spending Patterns & Anomaly Detection

In [ ]:
# ── Z-score based anomaly detection on Total Expenditure ─────────────────────
from scipy import stats

df_clean['Z_Score'] = np.abs(stats.zscore(df_clean['Total_Expenditure']))
anomalies = df_clean[df_clean['Z_Score'] > 1.5][['Month','Year','Total_Expenditure','Z_Score']].sort_values('Z_Score', ascending=False)

print('Unusual / High Expense Months (|Z-score| > 1.5):')
print(anomalies.to_string(index=False))

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_clean['Month'], df_clean['Total_Expenditure'], color='steelblue',
        linewidth=1.8, label='Monthly Expenditure')
ax.scatter(
    df_clean.loc[df_clean['Z_Score'] > 1.5, 'Month'],
    df_clean.loc[df_clean['Z_Score'] > 1.5, 'Total_Expenditure'],
    color='red', zorder=5, s=80, label='Anomaly (|Z|>1.5)'
)
mean_val = df_clean['Total_Expenditure'].mean()
std_val  = df_clean['Total_Expenditure'].std()
ax.axhline(mean_val, color='green', linestyle='--', linewidth=1, label=f'Mean ₹{mean_val:,.0f}')
ax.axhline(mean_val + 1.5*std_val, color='orange', linestyle=':', linewidth=1, label='±1.5σ')
ax.axhline(mean_val - 1.5*std_val, color='orange', linestyle=':', linewidth=1)
ax.set_title('Monthly Expenditure – Anomaly Detection (Z-Score Method)', fontweight='bold')
ax.set_ylabel('Total Expenditure (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.savefig('chart_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Rolling 3-month average for trend ─────────────────────────────────────────
df_clean['Rolling_3M'] = df_clean['Total_Expenditure'].rolling(3, min_periods=1).mean()
df_clean['Rolling_6M'] = df_clean['Total_Expenditure'].rolling(6, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_clean['Month'], df_clean['Total_Expenditure'], color='lightblue',
        linewidth=1.5, label='Monthly')
ax.plot(df_clean['Month'], df_clean['Rolling_3M'], color='steelblue',
        linewidth=2, label='3-Month Rolling Avg')
ax.plot(df_clean['Month'], df_clean['Rolling_6M'], color='navy',
        linewidth=2, linestyle='--', label='6-Month Rolling Avg')
ax.set_title('Spending Trend – Rolling Averages', fontweight='bold')
ax.set_ylabel('Expenditure (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax.legend()
plt.tight_layout()
plt.savefig('chart_rolling_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Year-over-Year Growth Analysis

In [ ]:
yoy = yearly_summary[['Total_Expenditure','Total_Income']].copy()
yoy['Exp_Growth_%'] = yoy['Total_Expenditure'].pct_change() * 100
yoy['Inc_Growth_%'] = yoy['Total_Income'].pct_change() * 100
print('Year-over-Year Growth:')
print(yoy.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(yoy.index))
width = 0.35
ax.bar(x - width/2, yoy['Exp_Growth_%'].fillna(0), width, label='Expenditure Growth %', color='steelblue')
ax.bar(x + width/2, yoy['Inc_Growth_%'].fillna(0), width, label='Income Growth %', color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(yoy.index)
ax.set_title('Year-over-Year Growth: Expenditure vs Income', fontweight='bold')
ax.set_ylabel('Growth (%)')
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.savefig('chart_yoy_growth.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Personal Financial Insights

In [ ]:
print('=' * 65)
print('       PERSONAL FINANCIAL INSIGHTS (2020–2025)')
print('=' * 65)

# 1. Average savings rate
avg_savings_rate = df_clean['Savings_Rate'].mean()
print(f'\n📈 1. Average Savings Rate: {avg_savings_rate:.1f}%')
if avg_savings_rate >= 20:
    print('   ✅ Excellent! You consistently save more than 20% of income.')
elif avg_savings_rate >= 10:
    print('   ⚠️  Good, but aim for 20%+ for stronger financial security.')
else:
    print('   ❌ Low savings rate. Consider cutting discretionary spending.')

# 2. Rent burden
avg_rent_pct = (df_clean['Rent'] / df_clean['Income'] * 100).mean()
print(f'\n🏠 2. Average Rent as % of Income: {avg_rent_pct:.1f}%')
if avg_rent_pct <= 30:
    print('   ✅ Rent is within the recommended 30% threshold.')
else:
    print('   ⚠️  Rent exceeds 30% of income — consider renegotiating or relocating.')

# 3. Discretionary spending trend
disc_first = df_clean[df_clean['Year']==2020]['Discretionary'].mean()
disc_last  = df_clean[df_clean['Year']==2025]['Discretionary'].mean()
disc_change = (disc_last - disc_first) / disc_first * 100
print(f'\n🛍️  3. Discretionary Spending Change (2020→2025): {disc_change:+.1f}%')
if disc_change > 15:
    print('   ⚠️  Discretionary spending has grown significantly. Review wants vs needs.')
else:
    print('   ✅ Discretionary spending is well controlled.')

# 4. Investment rate
avg_inv_rate = (df_clean['Investments'] / df_clean['Income'] * 100).mean()
print(f'\n💼 4. Average Investment Rate: {avg_inv_rate:.1f}%')
if avg_inv_rate >= 15:
    print('   ✅ Strong investment discipline! Wealth is being built consistently.')
elif avg_inv_rate >= 8:
    print('   ⚠️  Moderate investment rate. Try to increase to 15%+ of income.')
else:
    print('   ❌ Low investment rate. Prioritize long-term wealth building.')

# 5. EMI Impact
emi_months = (df_clean['EMI/Loans'] > 0).sum()
avg_emi = df_clean[df_clean['EMI/Loans']>0]['EMI/Loans'].mean()
print(f'\n💳 5. EMI/Loan: Active for {emi_months} months (avg ₹{avg_emi:,.0f}/month)')
if emi_months > 0:
    avg_emi_pct = (df_clean['EMI/Loans'] / df_clean['Income'] * 100).mean()
    print(f'   EMI as % of income (active period): {avg_emi_pct:.1f}%')
    if avg_emi_pct < 10:
        print('   ✅ EMI burden is manageable.')
    else:
        print('   ⚠️  EMI exceeds 10% of income. Monitor debt levels.')

# 6. Highest spending month
max_row = df_clean.loc[df_clean['Total_Expenditure'].idxmax()]
print(f'\n📅 6. Highest Spending Month: {max_row["Month"].strftime("%B %Y")} — ₹{max_row["Total_Expenditure"]:,.0f}')

# 7. Category that grew most YoY
cat_growth = (
    (df_clean[df_clean['Year']==2025][CAT_COLS].mean() -
     df_clean[df_clean['Year']==2020][CAT_COLS].mean()) /
     df_clean[df_clean['Year']==2020][CAT_COLS].mean() * 100
).sort_values(ascending=False)
print(f'\n📊 7. Category with Highest 5-Year Growth: {cat_growth.idxmax()} ({cat_growth.max():+.1f}%)')
print(f'   Category with Lowest  5-Year Growth: {cat_growth.idxmin()} ({cat_growth.min():+.1f}%)')

# 8. Income vs Expenditure check
months_over_income = (df_clean['Total_Expenditure'] > df_clean['Income']).sum()
print(f'\n⚠️  8. Months where spending exceeded income: {months_over_income}')

print('\n' + '='*65)
print('END OF INSIGHTS')
print('='*65)

## 12. Correlation Analysis

In [ ]:
corr_cols = CAT_COLS + ['Total_Expenditure', 'Income', 'Savings_Rate']
corr_matrix = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Correlation Matrix – All Financial Variables', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Final Summary Table

In [ ]:
print('='*70)
print('FINAL SUMMARY: KEY METRICS (2020–2025)')
print('='*70)

total_months   = len(df_clean)
total_exp      = df_clean['Total_Expenditure'].sum()
total_income   = df_clean['Income'].sum()
total_savings  = df_clean['Savings'].sum()
total_inv      = df_clean['Investments'].sum()
avg_monthly_exp = df_clean['Total_Expenditure'].mean()
avg_savings_r  = df_clean['Savings_Rate'].mean()

summary_dict = {
    'Total Months Analyzed'        : total_months,
    'Total Income (₹)'             : f'{total_income:,.0f}',
    'Total Expenditure (₹)'        : f'{total_exp:,.0f}',
    'Total Savings (₹)'            : f'{total_savings:,.0f}',
    'Total Investments (₹)'        : f'{total_inv:,.0f}',
    'Avg Monthly Expenditure (₹)'  : f'{avg_monthly_exp:,.0f}',
    'Avg Savings Rate (%)'         : f'{avg_savings_r:.1f}',
    'Highest Spending Category'    : cat_totals.idxmax(),
    'Lowest  Spending Category'    : cat_totals.idxmin(),
}

for k, v in summary_dict.items():
    print(f'  {k:<40} {v}')

print('='*70)
print('Analysis complete ✅')